# Structural Entanglement Unlearning — Kaggle pilot

This notebook is deliberately gated. It pins the repository commit and Python dependencies, checks the live Kaggle resources, validates the chosen dataset and model access, and runs a **one-update smoke experiment using the selected model** before enabling main conditions. It never assumes a Kaggle GPU type, quota, or remaining runtime. Do not enable main runs until the smoke report says `passed` and you have reviewed the measured time/memory and research-use permissions.

**Persistence:** write all artifacts under `/kaggle/working/pilot-artifacts`. Use Kaggle **Save Version** after the smoke test and after each completed condition (or before a planned stop). For recovery, attach the prior notebook version/output as an input, copy its `pilot-artifacts` directory back into `/kaggle/working`, and rerun the same cells/config. This workflow does not rely on an assumed external drive or storage service.

Run sections in order. Section 7 is explicitly gated by `START_MAIN_EXPERIMENT = False`; changing it is a deliberate user action.

## 1. Setup: clone exact project revision and install pinned dependencies

Run once in a fresh Kaggle session. Publish a commit containing both this notebook and `requirements-kaggle.lock`. Set `REPOSITORY_COMMIT` to that published commit's full SHA, then publish the pin update too (the pinned commit must already contain both files). Installing the lock may replace Kaggle's preinstalled PyTorch; restart the notebook kernel if pip reports that a loaded library changed, then rerun from section 1. If the pinned PyTorch wheel does not expose CUDA in section 2, stop and adapt the lock to a CUDA-enabled wheel compatible with the Kaggle driver before continuing.

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/ArsalanKaleem/structurale-ntanglement-unlearning.git"
REPOSITORY_COMMIT = "89d9058851ed30b093916c7fb3cb91a9667142d5"
PROJECT_DIR = Path("/kaggle/working/structural-entanglement-unlearning")
ARTIFACT_ROOT = Path("/kaggle/working/pilot-artifacts")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

if len(REPOSITORY_COMMIT) != 40 or any(c not in "0123456789abcdefABCDEF" for c in REPOSITORY_COMMIT):
    raise ValueError("Set REPOSITORY_COMMIT to the published 40-character commit containing this notebook and lock")
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPOSITORY_COMMIT], cwd=PROJECT_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=PROJECT_DIR, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip() == REPOSITORY_COMMIT

subprocess.run(
    ["python", "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-kaggle.lock")],
    check=True,
)
subprocess.run(
    ["python", "-m", "pip", "install", "--no-deps", "-e", str(PROJECT_DIR)],
    check=True,
)
print("Pinned checkout:", REPOSITORY_COMMIT)
print("Project:", PROJECT_DIR)
print("Artifacts:", ARTIFACT_ROOT)

### Optional: restore artifacts from a previously saved Kaggle output version

Run after setup and before validation or experiments in a new session. Set the source to the `pilot-artifacts` directory under the attached prior Kaggle output input. Leave `None` for a fresh run.

In [ ]:
import shutil
RECOVERY_ARTIFACTS = None  # Example: Path('/kaggle/input/previous-pilot-version/pilot-artifacts')
if RECOVERY_ARTIFACTS is not None:
    recovery_source = Path(RECOVERY_ARTIFACTS)
    if not recovery_source.is_dir():
        raise FileNotFoundError(f"Prior saved artifacts not found: {recovery_source}")
    shutil.copytree(recovery_source, ARTIFACT_ROOT, dirs_exist_ok=True)
    print("Restored Kaggle output from:", recovery_source)
else:
    print("Fresh run; no prior Kaggle output restored.")

## 2. Inspect this session's actual Python, PyTorch, CUDA, GPU, memory, disk, and clock

This reports detected resources only; it does not infer a quota or time allowance. Stop if CUDA is unavailable or if the selected model cannot fit based on the observed smoke-test peak.

In [ ]:
import json
import platform
import shutil
import sys
import time
import torch
import transformers
from importlib.metadata import version

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}; total memory={props.total_memory / 2**30:.2f} GiB; capability={props.major}.{props.minor}")
        print("Current free/total:", torch.cuda.mem_get_info(index))
else:
    raise RuntimeError("No CUDA GPU is visible. Enable a Kaggle GPU accelerator before proceeding.")
print("Working disk free bytes:", shutil.disk_usage("/kaggle/working").free)
try:
    print("Session uptime seconds:", float(Path("/proc/uptime").read_text().split()[0]))
except OSError:
    print("Session uptime: unavailable on this runtime")
print("Kaggle-related environment keys:", sorted(k for k in os.environ if k.startswith("KAGGLE_")))
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

with open(ARTIFACT_ROOT / "session_environment.json", "w") as f:
    json.dump({
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "cuda_build": torch.version.cuda,
        "gpu_count": torch.cuda.device_count(),
        "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "working_disk_free_bytes": shutil.disk_usage("/kaggle/working").free,
        "device": [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())],
        "package_versions": {name: version(name) for name in ("torch", "transformers", "tokenizers", "datasets", "accelerate", "numpy", "pandas", "scikit-learn", "PyYAML", "tqdm", "pytest", "huggingface-hub")},
    }, f, indent=2)

## 3. Select inputs and verify accessibility and research-use permissions

Set the mounted Kaggle dataset JSONL path, a descriptive immutable dataset version, and the exact Hugging Face model commit SHA. The notebook queries the Hub for model access, commit, and declared license metadata; it cannot decide whether that license or the dataset terms permit this research. Read the model license and dataset source/license yourself and set both review confirmations to `True` only after checking. Do not proceed on a gated/private model unless Kaggle credentials have been configured securely and access is authorized.

In [ ]:
import copy
import hashlib
import requests
import yaml

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
DATASET_JSONL = PROJECT_DIR / "data" / "fictional_authors_v1.jsonl"
DATASET_VERSION = "fictional-authors-v1-sha256-5644e12fbc7e"
DATASET_SOURCE = "Synthetic; generated by scripts/00_generate_fictional_dataset.py (seed 20261013) in this repository"
DATASET_LICENSE = "MIT (repository license); fully invented text, no third-party content"
MODEL_LICENSE_REVIEWED_FOR_THIS_USE = False
DATASET_RIGHTS_REVIEWED_FOR_THIS_USE = False

if len(MODEL_REVISION) != 40 or any(c not in "0123456789abcdefABCDEF" for c in MODEL_REVISION):
    raise ValueError("Set MODEL_REVISION to the exact 40-character model commit SHA")
if not DATASET_JSONL.is_file():
    raise FileNotFoundError(f"Dataset JSONL is not accessible: {DATASET_JSONL}")
if DATASET_VERSION.startswith("REPLACE") or DATASET_SOURCE.startswith("REPLACE") or DATASET_LICENSE.startswith("REPLACE"):
    raise ValueError("Fill dataset version, source, and license before running")

model_response = requests.get(
    f"https://huggingface.co/api/models/{MODEL_ID}/revision/{MODEL_REVISION}", timeout=30
)
model_response.raise_for_status()
model_info = model_response.json()
print("Hub model:", model_info.get("id"))
print("Resolved model SHA:", model_info.get("sha"))
print("Declared model license:", (model_info.get("cardData") or {}).get("license"))
print("Model card:", f"https://huggingface.co/{MODEL_ID}")
assert model_info.get("sha", "").lower() == MODEL_REVISION.lower(), "Hub returned a different model revision"
dataset_hasher = hashlib.sha256()
with DATASET_JSONL.open("rb") as dataset_file:
    for chunk in iter(lambda: dataset_file.read(1024 * 1024), b""):
        dataset_hasher.update(chunk)
dataset_sha256 = dataset_hasher.hexdigest()
print("Dataset path:", DATASET_JSONL)
print("Dataset version/source/license:", DATASET_VERSION, DATASET_SOURCE, DATASET_LICENSE)
print("Dataset SHA-256:", dataset_sha256)
if not MODEL_LICENSE_REVIEWED_FOR_THIS_USE or not DATASET_RIGHTS_REVIEWED_FOR_THIS_USE:
    raise PermissionError("Review and confirm model and dataset research-use permissions in this cell")

PILOT_CONFIG = {
    "experiment": {
        "name": "kaggle_pilot", "seed": 17, "output_dir": str(ARTIFACT_ROOT / "main"),
        # Keep only the checkpoint needed to resume; delete it once a condition completes.
        "checkpoint_retention": "resume_only",
    },
    "model": {"name_or_path": MODEL_ID, "revision": MODEL_REVISION, "device": "cuda", "max_length": 256},
    "data": {
        "input_jsonl": str(DATASET_JSONL), "version": DATASET_VERSION,
        "max_forget_examples": 64, "max_forget_update_examples": 64,
        "max_retain_examples": 64, "max_control_examples": 64,
        "max_control_update_examples": 64, "max_retain_update_examples": 64,
    },
    "unlearning": {
        "method": "ga", "learning_rate": 1e-5, "max_steps_per_request": 10,
        "batch_size": 1, "sequence_length": 16, "gradient_accumulation_steps": 1,
    },
    "evaluation": {
        "primary_structure_metric": "cosine_separation", "prompt_template": "{text}",
        "fixed_layer": -1, "max_eval_examples_per_split": 64,
        "generation_max_new_tokens": 32,
    },
}
config_path = ARTIFACT_ROOT / "pilot.yaml"
config_path.write_text(yaml.safe_dump(PILOT_CONFIG, sort_keys=False))
(ARTIFACT_ROOT / "input_provenance.json").write_text(json.dumps({
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "declared_model_license": (model_info.get("cardData") or {}).get("license"),
    "model_license_reviewed_for_this_use": MODEL_LICENSE_REVIEWED_FOR_THIS_USE,
    "dataset_version": DATASET_VERSION, "dataset_source": DATASET_SOURCE,
    "dataset_license": DATASET_LICENSE,
    "dataset_rights_reviewed_for_this_use": DATASET_RIGHTS_REVIEWED_FOR_THIS_USE,
    "dataset_sha256": dataset_sha256,
}, indent=2))
print("Pilot config written to:", config_path)

# Frozen protocol (docs/protocol.md): 5 seeds; shuffled order for all seeds, original
# order for seed 17 only. Replacement seeds 41, 43 are used only for excluded seeds.
SEEDS = [17, 23, 29, 31, 37]
ORIGINAL_ORDER_SEEDS = {17}


def seed_root(seed):
    return ARTIFACT_ROOT / "main" / f"seed_{seed}"


def seed_config_path(seed):
    return ARTIFACT_ROOT / f"pilot_seed_{seed}.yaml"


for seed in SEEDS:
    seed_config = copy.deepcopy(PILOT_CONFIG)
    seed_config["experiment"]["seed"] = seed
    seed_config["experiment"]["output_dir"] = str(seed_root(seed))
    seed_config_path(seed).write_text(yaml.safe_dump(seed_config, sort_keys=False))
print("Per-seed configs:", [str(seed_config_path(seed)) for seed in SEEDS])

## 4. Data validation and software tests

The loader requires separate held-out `forget`/`retain`/`control` and update-only `forget_update`/`retain_update`/`control_update` splits, enough rows for the configured sequence, and no exact normalized text duplicates across splits. This catches only exact duplicates—not paraphrases, semantic leakage, pretraining contamination, or licensing issues. Run tests and validate the real mounted dataset before downloading model weights.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)
subprocess.run([
    sys.executable, "scripts/01_prepare_data.py",
    "--input", str(DATASET_JSONL),
    "--output", str(ARTIFACT_ROOT / "validated-data"),
], cwd=PROJECT_DIR, check=True)
summary = json.loads((ARTIFACT_ROOT / "validated-data" / "split_summary.json").read_text())
print("Validated split counts:", summary)
required_splits = {"forget", "retain", "control", "forget_update", "control_update", "retain_update"}
if not required_splits.issubset(summary) or min(summary[name] for name in required_splits) < 1:
    raise ValueError("Dataset needs at least one row in every held-out and update-only split")
if summary["forget_update"] < 1 or summary["control_update"] < 1 or summary["retain_update"] < 1:
    raise ValueError("Dataset has too few update requests for the smoke test")
if summary["forget"] < 2:
    raise ValueError("The primary cosine-separation metric needs at least two held-out forget examples")
required_main_requests = int(PILOT_CONFIG["unlearning"]["sequence_length"])
for split_name in ("forget_update", "control_update", "retain_update"):
    if summary[split_name] < required_main_requests:
        raise ValueError(f"{split_name} has {summary[split_name]} rows; main run requires {required_main_requests}")

## 5. Tiny end-to-end smoke test — mandatory gate

This uses the selected pinned model and one update per condition on the first valid requests, with the same dataset, template, and evaluation rows. It saves a separate smoke base checkpoint, then runs no-update, forget GA, control GA, retain GA, and paired analysis. It is intentionally small but still downloads/loads the selected model. **Do not run this unless GPU, storage, access, and permissions checks above passed.** A failure stops the notebook; investigate before any main experiment.

In [ ]:
import shutil
import copy
import runpy
import time
import pandas as pd

SMOKE_ROOT = ARTIFACT_ROOT / "smoke"
SMOKE_CONFIG = copy.deepcopy(PILOT_CONFIG)
SMOKE_CONFIG["experiment"]["name"] = "kaggle_smoke"
SMOKE_CONFIG["experiment"]["output_dir"] = str(SMOKE_ROOT)
SMOKE_CONFIG["unlearning"]["sequence_length"] = 1
for key in ("max_forget_update_examples", "max_retain_examples",
            "max_control_examples", "max_control_update_examples", "max_retain_update_examples"):
    SMOKE_CONFIG["data"][key] = 1
SMOKE_CONFIG["data"]["max_forget_examples"] = 2
SMOKE_CONFIG["evaluation"]["max_eval_examples_per_split"] = 2
smoke_config_path = ARTIFACT_ROOT / "smoke.yaml"
smoke_config_path.write_text(yaml.safe_dump(SMOKE_CONFIG, sort_keys=False))
SMOKE_REPORT_PATH = ARTIFACT_ROOT / "smoke-report.json"
SMOKE_REPORT_PATH.unlink(missing_ok=True)

def run_script_timed(script, args):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()
    old_argv = sys.argv
    try:
        sys.argv = [str(PROJECT_DIR / script), *args]
        runpy.run_path(str(PROJECT_DIR / script), run_name="__main__")
    finally:
        sys.argv = old_argv
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
    measurement = {"script": script, "elapsed_seconds": elapsed, "peak_allocated_bytes": int(peak)}
    measurements_dir = ARTIFACT_ROOT / "resource-measurements"
    measurements_dir.mkdir(parents=True, exist_ok=True)
    json_path = measurements_dir / "measurements.json"
    previous = json.loads(json_path.read_text()) if json_path.exists() else []
    previous.append(measurement)
    json_path.write_text(json.dumps(previous, indent=2))
    pd.DataFrame(previous).to_csv(measurements_dir / "measurements.csv", index=False)
    return measurement

smoke_measurements = []
smoke_measurements.append(run_script_timed("scripts/02_run_baseline.py", ["--config", str(smoke_config_path)]))
for condition in ("forget", "control", "retain"):
    smoke_measurements.append(run_script_timed(
        "scripts/03_run_sequential.py",
        ["--config", str(smoke_config_path), "--condition", condition, "--order", "original"],
    ))
run_script_timed("scripts/06_analyze.py", ["--input", str(SMOKE_ROOT), "--output", str(SMOKE_ROOT / "analysis")])
smoke_peak_bytes = max(row["peak_allocated_bytes"] for row in smoke_measurements)
update_seconds = [row["elapsed_seconds"] for row in smoke_measurements[1:]]
measured_update_seconds = sum(update_seconds) / len(update_seconds)
requests_per_full_condition = int(PILOT_CONFIG["unlearning"]["sequence_length"])
estimated_update_wall_seconds = measured_update_seconds * requests_per_full_condition
main_condition_runs = sum(3 * (2 if seed in ORIGINAL_ORDER_SEEDS else 1) for seed in SEEDS)
estimated_main_wall_seconds = estimated_update_wall_seconds * main_condition_runs
smoke_report = {
    "status": "passed",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_sha256": dataset_sha256,
    "pilot_config_sha256": hashlib.sha256(json.dumps(PILOT_CONFIG, sort_keys=True).encode()).hexdigest(),
    "measurements": smoke_measurements,
    "smoke_peak_allocated_gpu_bytes": smoke_peak_bytes,
    "measured_mean_one_request_condition_seconds": measured_update_seconds,
    "configured_main_requests_per_condition": requests_per_full_condition,
    "linear_update_time_projection_seconds_per_condition": estimated_update_wall_seconds,
    "main_condition_runs": main_condition_runs,
    "linear_update_time_projection_seconds_main_total": estimated_main_wall_seconds,
    "projection_note": "Rough linear projection from measured one-request condition wall time multiplied by configured request count; this repeats fixed smoke setup/evaluation costs mathematically and is not a bound or quota guarantee. Full-scale evaluation and download variability are not measured.",
    "memory_note": "Observed peak for smoke settings only; larger eval caps, sequence lengths, or batches can require more memory.",
}
SMOKE_REPORT_PATH.write_text(json.dumps(smoke_report, indent=2))
print(json.dumps(smoke_report, indent=2))
if smoke_peak_bytes == 0:
    raise RuntimeError("Smoke did not report GPU memory use; do not start main runs")
shutil.rmtree(SMOKE_ROOT / "base_model", ignore_errors=True)  # metrics and reports are kept
print("Projected main GPU-hours (protocol ceiling 25):", round(estimated_main_wall_seconds / 3600, 2))
print("SMOKE GATE PASSED. Save a Kaggle Notebook Version now and review memory/runtime before enabling main conditions.")

## 6. Main-run gate

Main runs refuse to start until the smoke report says `passed`, matches the current model, data, and configuration, and the explicit switch is changed. Check the projected GPU-hours in the smoke report against the protocol's compute ceiling first; any allowed reduction must be recorded in `docs/protocol.md` before enabling main runs.

In [ ]:
START_MAIN_EXPERIMENT = False  # Change only after reviewing the smoke report and frozen protocol.
SMOKE_REPORT_PATH = ARTIFACT_ROOT / "smoke-report.json"
smoke_report = json.loads(SMOKE_REPORT_PATH.read_text())
if smoke_report.get("status") != "passed":
    raise RuntimeError("Smoke test did not pass; main experiment is blocked")
if (smoke_report.get("model_revision") != MODEL_REVISION
        or smoke_report.get("dataset_sha256") != dataset_sha256
        or smoke_report.get("pilot_config_sha256") != hashlib.sha256(json.dumps(PILOT_CONFIG, sort_keys=True).encode()).hexdigest()):
    raise RuntimeError("Smoke evidence does not match the current model, data, and main configuration")
if not START_MAIN_EXPERIMENT:
    print("Main experiment remains disabled. Set START_MAIN_EXPERIMENT=True after approval.")
else:
    print("Main experiment enabled for seeds", SEEDS)

## 7. Main runs, one seed at a time

For each seed this saves the seed's base checkpoint and no-update baseline, then runs forget, control, and retain GA (shuffled order; plus original order for seed 17). Conditions keep only the checkpoint needed to resume and delete it on completion; the seed's base checkpoint is deleted once all its conditions are complete, so disk use stays at roughly one base plus two step checkpoints. Completed seeds are skipped when the cell is rerun after an interruption. Save a Kaggle Notebook Version after each seed.

In [ ]:
if not START_MAIN_EXPERIMENT:
    raise RuntimeError("Main experiment is disabled")
import shutil

MIN_FREE_DISK_BYTES = 8 * 1024**3


def seed_runs(seed):
    orders = ["shuffled"] + (["original"] if seed in ORIGINAL_ORDER_SEEDS else [])
    return [(condition, order) for order in orders for condition in ("forget", "control", "retain")]


def seed_complete(seed):
    names = ["condition_no_update_original"] + [f"condition_{c}_{o}" for c, o in seed_runs(seed)]
    for name in names:
        run_json = seed_root(seed) / name / "run.json"
        if not run_json.is_file() or json.loads(run_json.read_text()).get("status") != "complete":
            return False
    return True


for seed in SEEDS:
    if seed_complete(seed):
        print(f"Seed {seed}: already complete, skipping")
        continue
    free = shutil.disk_usage("/kaggle/working").free
    if free < MIN_FREE_DISK_BYTES:
        raise RuntimeError(f"Only {free / 1024**3:.1f} GiB free; need 8 GiB before seed {seed}")
    config_arg = ["--config", str(seed_config_path(seed))]
    print(f"Seed {seed}: baseline", run_script_timed("scripts/02_run_baseline.py", config_arg))
    for condition, order in seed_runs(seed):
        measurement = run_script_timed(
            "scripts/03_run_sequential.py", [*config_arg, "--condition", condition, "--order", order]
        )
        print(f"Seed {seed}: completed {condition} {order}", measurement)
    if not seed_complete(seed):
        raise RuntimeError(f"Seed {seed} has incomplete conditions; inspect run.json files")
    shutil.rmtree(seed_root(seed) / "base_model")
    print(f"Seed {seed} complete. Free disk: {shutil.disk_usage('/kaggle/working').free / 1024**3:.1f} GiB")
    print("Save a Kaggle Notebook Version now.")


## 8. Completion check

Confirms every seed has a complete no-update baseline and all matched forget/control/retain runs (control and retain streams use the same request count, optimizer steps, and nearest-token-length pairing as forget GA) before analysis.

In [ ]:
if not START_MAIN_EXPERIMENT:
    raise RuntimeError("Main experiment is disabled")
incomplete = [seed for seed in SEEDS if not seed_complete(seed)]
if incomplete:
    raise RuntimeError(f"Incomplete seeds: {incomplete}; rerun section 7")
print("All seeds complete:", SEEDS)


## 9. Evaluation and analysis

Metrics are produced at baseline and each request checkpoint with fixed examples/prompts; this section validates outputs and writes paired effect/uncertainty summaries. The analysis requires complete matching runs and multiple seeds for useful uncertainty. With one seed, the interval and standardized effect are deliberately omitted. No output constitutes evidence for inversion without the preregistered rule and utility checks.

In [ ]:
if not START_MAIN_EXPERIMENT:
    raise RuntimeError("Main experiment is disabled")
for seed in SEEDS:
    run_script_timed("scripts/04_evaluate.py", ["--config", str(seed_config_path(seed))])
run_script_timed(
    "scripts/06_analyze.py",
    ["--input", str(ARTIFACT_ROOT / "main"), "--output", str(ARTIFACT_ROOT / "main" / "analysis")],
)
print("Run metadata, metrics, and analysis:", ARTIFACT_ROOT / "main")
print("Save a Kaggle Notebook Version to persist the completed outputs.")

## Failure recovery

1. A failed update records `status: failed` and its exception in that condition's `run.json`; the exception is not hidden. Keep the artifacts and inspect `run.log`.
2. Before ending a session, use Kaggle **Save Version** so `/kaggle/working/pilot-artifacts` becomes a versioned notebook output. For a new session, add that prior version as an input, locate its `pilot-artifacts` directory under `/kaggle/input`, and copy it to `/kaggle/working/pilot-artifacts` before running setup.
3. Rerun setup and the exact same configuration/condition/order. Completed step checkpoints are restored; if a checkpoint exists but its final metrics row was interrupted, that row is recalculated without repeating the update.
4. Do not change input data, config, model revision, or prompt for an existing output directory. The run fingerprint rejects a mismatch. For a deliberate new experiment, use a new output directory.
5. If the smoke test fails or GPU memory is inadequate, do not set the main-run gate. Reduce the protocol's prespecified resource demands, use a model that fits, or select another available GPU, then repeat and document a fresh smoke test. This notebook does not assume Kaggle quota or session duration.